# 🤖 Agent Basics

**Build autonomous AI agents that can plan and execute**

---

## 📋 Overview

**What you'll learn:**
- What is an AI agent?
- Agent architecture
- Planning and reasoning
- Building your first agent
- Agent frameworks comparison

**Time estimate:** ⏱️ 60 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List, Any

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 What is an AI Agent?

### Function Calling vs Agents:

**Function Calling (Tool Use):**
```
User: "Book a flight to Paris"
LLM: Calls search_flights()
     Calls book_flight()
     Done

→ Reactive: Responds to explicit requests
```

**Agent:**
```
User: "Plan a vacation to Europe"
Agent: 1. What does user want? (Reasoning)
       2. Need to check budget (Planning)
       3. Compare cities (Action)
       4. Check weather (Action)
       5. Find flights (Action)
       6. Book hotels (Action)
       7. Create itinerary (Action)

→ Proactive: Breaks down goals autonomously
```

### Agent Components:

**1. Brain (LLM)**
- Reasoning
- Planning
- Decision making

**2. Tools**
- APIs
- Databases
- External systems

**3. Memory**
- Short-term (conversation)
- Long-term (knowledge base)

**4. Planning**
- Goal decomposition
- Task sequencing
- Error recovery

### Agent Loop:

```
1. Observe: Get current state
2. Think: What should I do?
3. Act: Execute action
4. Reflect: Did it work?
5. Repeat until goal achieved
```

## 🏗️ Simple Agent Architecture

In [ ]:
class SimpleAgent:
    """Basic autonomous agent."""
    
    def __init__(self, tools: List[Dict], max_iterations: int = 10):
        self.tools = tools
        self.max_iterations = max_iterations
        self.memory = []  # Conversation history
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    def run(self, goal: str) -> str:
        """Run agent to achieve goal."""
        
        print(f"🤖 Agent Goal: {goal}")
        print("="*60)
        
        # Initialize with system prompt
        self.memory = [
            {
                "role": "system",
                "content": """You are an autonomous AI agent. Your job is to:
1. Break down the user's goal into steps
2. Use available tools to complete each step
3. Reflect on results and adjust your plan
4. Continue until the goal is achieved

Think step-by-step and explain your reasoning."""
            },
            {"role": "user", "content": goal}
        ]
        
        for iteration in range(self.max_iterations):
            print(f"\n🔄 Iteration {iteration + 1}")
            
            # Think & Act
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=self.memory,
                tools=self.tools,
                tool_choice="auto"
            )
            
            response_message = response.choices[0].message
            
            # Agent's thoughts
            if response_message.content:
                print(f"💭 Thought: {response_message.content}")
            
            # Check if done
            if not response_message.tool_calls:
                print("✅ Goal achieved!")
                return response_message.content
            
            # Add to memory
            self.memory.append(response_message)
            
            # Execute actions
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"🔧 Action: {function_name}({function_args})")
                
                # Execute (mock)
                result = self._execute_tool(function_name, function_args)
                print(f"   Result: {result}")
                
                # Add result to memory
                self.memory.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
        
        return "Max iterations reached"
    
    def _execute_tool(self, name: str, args: Dict) -> Dict:
        """Execute a tool (mock implementation)."""
        # In production, call real tools
        mock_results = {
            "search": {"results": ["Result 1", "Result 2"]},
            "calculate": {"answer": 42},
            "get_weather": {"temp": 20, "condition": "Sunny"},
        }
        
        return mock_results.get(name, {"status": "executed"})

# Example tools
agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the web for information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform mathematical calculations",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    }
]

# Run agent
agent = SimpleAgent(tools=agent_tools)
result = agent.run("Find the population of Paris and calculate what 10% of it is")

print(f"\n" + "="*60)
print(f"\n📊 Final Result:\n{result}")

## 🧠 Agent with Planning

In [ ]:
class PlanningAgent:
    """Agent that creates a plan before executing."""
    
    def __init__(self, tools: List[Dict]):
        self.tools = tools
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    def create_plan(self, goal: str) -> List[str]:
        """Create a step-by-step plan."""
        
        print(f"\n📋 Creating Plan...\n")
        
        planning_prompt = f"""Break down this goal into concrete steps:

Goal: {goal}

Available tools: {', '.join([t['function']['name'] for t in self.tools])}

Create a numbered list of steps. Each step should use one tool.
Format:
1. [tool_name] Description
2. [tool_name] Description
...

Plan:"""
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": planning_prompt}],
            temperature=0
        )
        
        plan_text = response.choices[0].message.content
        print(plan_text)
        
        # Parse plan
        steps = [line.strip() for line in plan_text.split('\n') if line.strip() and line[0].isdigit()]
        
        return steps
    
    def execute_plan(self, steps: List[str]) -> str:
        """Execute each step of the plan."""
        
        print(f"\n🚀 Executing Plan...\n")
        
        results = []
        
        for i, step in enumerate(steps, 1):
            print(f"Step {i}: {step}")
            # Execute step (simplified)
            result = {"step": i, "status": "completed"}
            results.append(result)
            print(f"  ✅ Completed\n")
        
        return "All steps completed successfully"

# Example
planning_agent = PlanningAgent(tools=agent_tools)

goal = "Research the weather in Paris and London, then tell me which is warmer"
print(f"🎯 Goal: {goal}")

plan = planning_agent.create_plan(goal)
result = planning_agent.execute_plan(plan)

print(f"\n✅ {result}")

## 🔄 Agent Frameworks Comparison

In [ ]:
import pandas as pd

frameworks = pd.DataFrame([
    {
        'Framework': 'LangChain',
        'Ease of Use': '⭐⭐⭐',
        'Features': 'Very comprehensive',
        'Best For': 'Complex agents, RAG',
        'Learning Curve': 'Medium',
    },
    {
        'Framework': 'LlamaIndex',
        'Ease of Use': '⭐⭐⭐⭐',
        'Features': 'Data-focused',
        'Best For': 'Data agents, indexing',
        'Learning Curve': 'Easy',
    },
    {
        'Framework': 'AutoGPT',
        'Ease of Use': '⭐⭐',
        'Features': 'Autonomous',
        'Best For': 'Long-running tasks',
        'Learning Curve': 'Hard',
    },
    {
        'Framework': 'CrewAI',
        'Ease of Use': '⭐⭐⭐⭐',
        'Features': 'Multi-agent',
        'Best For': 'Team of agents',
        'Learning Curve': 'Easy',
    },
    {
        'Framework': 'Custom (OpenAI)',
        'Ease of Use': '⭐⭐⭐⭐⭐',
        'Features': 'Full control',
        'Best For': 'Production, custom needs',
        'Learning Curve': 'Easy',
    },
])

print("🔄 Agent Frameworks\n")
print(frameworks.to_string(index=False))

print("\n💡 Recommendation:")
print("  - Learning: Start with custom OpenAI implementation")
print("  - Production: LangChain or custom")
print("  - Multi-agent: CrewAI")
print("  - Data-heavy: LlamaIndex")

## ✅ Summary

### Agent vs Tool Use:

| Aspect | Tool Use | Agent |
|--------|----------|-------|
| **Autonomy** | Reactive | Proactive |
| **Planning** | None | Yes |
| **Complexity** | Simple | Complex |
| **Iterations** | 1-3 | 5-20+ |
| **Use Case** | Direct tasks | Open-ended goals |

### Agent Architecture:

```python
class Agent:
    def __init__(self):
        self.brain = LLM()       # Reasoning
        self.tools = [...]       # Actions
        self.memory = []         # Context
        self.planner = Planner() # Planning
    
    def run(self, goal):
        plan = self.planner.create_plan(goal)
        
        for step in plan:
            # Observe
            state = self.get_state()
            
            # Think
            action = self.brain.decide(state, step)
            
            # Act
            result = self.execute(action)
            
            # Remember
            self.memory.append(result)
            
            # Reflect
            if self.is_goal_achieved():
                break
```

### Agent Loop:

```
User Goal → Plan → Observe → Think → Act → Reflect
                      ↑                        |
                      └────────────────────────┘
                         (Repeat until done)
```

### When to Use Agents:

✅ **Use agents for:**
- Open-ended goals
- Multi-step tasks
- Research/exploration
- Decision making
- Autonomous workflows

❌ **Don't use agents for:**
- Simple queries
- Single tool calls
- Deterministic tasks
- Real-time responses (too slow)

### Best Practices:

**1. Clear System Prompt**
```python
system_prompt = """
You are an autonomous agent.
Your goal: {goal}
Your tools: {tools}
Your approach:
1. Break down the goal
2. Use tools to gather info
3. Reason about results
4. Adjust plan as needed
"""
```

**2. Iteration Limits**
```python
max_iterations = 10  # Prevent infinite loops
```

**3. Memory Management**
```python
# Keep memory concise
if len(memory) > 20:
    memory = summarize(memory[:10]) + memory[10:]
```

**4. Error Recovery**
```python
if tool_fails:
    # Try alternative approach
    # Or ask for help
    # Or gracefully fail
```

### Simple Agent Template:

```python
class Agent:
    def run(self, goal):
        memory = [{"role": "user", "content": goal}]
        
        for i in range(max_iterations):
            # Think & Act
            response = llm(memory, tools=tools)
            
            # Done?
            if no_tool_calls(response):
                return response
            
            # Execute tools
            results = execute_tools(response.tool_calls)
            
            # Update memory
            memory.extend([response, results])
        
        return "Goal not achieved"
```

### Next: `07_agents_tools/04_react_agents.ipynb`